#Worksheet8

A Comprehensive Guide on Continuous Bag of Words Model.

In [ ]:
import math
import pandas as pd

# Documents
docs = {
    "D1": "The Cat chased the Mouse.",
    "D2": "The dog Barked at the Cat.",
    "D3": "The mouse ran away from the Cat."
}

# 🔹 Step 1: Preprocessing
processed_docs = {}

for key, text in docs.items():
    text = text.lower()                       # lowercase
    text = text.replace('.', '')              # remove punctuation
    tokens = text.split()                     # tokenize
    processed_docs[key] = tokens

print("Tokens:")
for k,v in processed_docs.items():
    print(k, ":", v)

Tokens:
D1 : ['the', 'cat', 'chased', 'the', 'mouse']
D2 : ['the', 'dog', 'barked', 'at', 'the', 'cat']
D3 : ['the', 'mouse', 'ran', 'away', 'from', 'the', 'cat']


Step 2: Vocabulary

In [ ]:
vocab = sorted(set(word for doc in processed_docs.values() for word in doc))
print("\nVocabulary:", vocab)


Vocabulary: ['at', 'away', 'barked', 'cat', 'chased', 'dog', 'from', 'mouse', 'ran', 'the']


Step 3: Term Frequency (TF)

In [ ]:
tf = {}

for doc_name, words in processed_docs.items():
    tf[doc_name] = {}
    total_words = len(words)

    for word in vocab:
        tf[doc_name][word] = words.count(word) / total_words

tf_df = pd.DataFrame(tf).T
print("\nTF Table:")
print(tf_df)


TF Table:
          at      away    barked       cat  chased       dog      from  \
D1  0.000000  0.000000  0.000000  0.200000     0.2  0.000000  0.000000   
D2  0.166667  0.000000  0.166667  0.166667     0.0  0.166667  0.000000   
D3  0.000000  0.142857  0.000000  0.142857     0.0  0.000000  0.142857   

       mouse       ran       the  
D1  0.200000  0.000000  0.400000  
D2  0.000000  0.000000  0.333333  
D3  0.142857  0.142857  0.285714  


Step 3.1: Normalized TF

In [ ]:
tf_norm = {}

for doc_name, words in processed_docs.items():
    tf_norm[doc_name] = {}

    for word in vocab:
        count = words.count(word)
        if count > 0:
            tf_norm[doc_name][word] = 1 + math.log10(count)
        else:
            tf_norm[doc_name][word] = 0

tf_norm_df = pd.DataFrame(tf_norm).T
print("\nNormalized TF:")
print(tf_norm_df)


Normalized TF:
     at  away  barked  cat  chased  dog  from  mouse  ran      the
D1  0.0   0.0     0.0  1.0     1.0  0.0   0.0    1.0  0.0  1.30103
D2  1.0   0.0     1.0  1.0     0.0  1.0   0.0    0.0  0.0  1.30103
D3  0.0   1.0     0.0  1.0     0.0  0.0   1.0    1.0  1.0  1.30103


Step 4: IDF

In [ ]:
N = len(processed_docs)
idf = {}

for word in vocab:
    df = sum(1 for doc in processed_docs.values() if word in doc)
    idf[word] = math.log10(N / (1 + df))

idf_df = pd.DataFrame.from_dict(idf, orient='index', columns=['IDF'])
print("\nIDF:")
print(idf_df)


IDF:
             IDF
at      0.176091
away    0.176091
barked  0.176091
cat    -0.124939
chased  0.176091
dog     0.176091
from    0.176091
mouse   0.000000
ran     0.176091
the    -0.124939


Step 5: TF-IDF

In [ ]:
tfidf = {}

for doc_name in processed_docs.keys():
    tfidf[doc_name] = {}

    for word in vocab:
        tfidf[doc_name][word] = tf[doc_name][word] * idf[word]

tfidf_df = pd.DataFrame(tfidf).T
print("\nTF-IDF Matrix:")
print(tfidf_df)

#1. What is the intuition behind using TF–IDF instead of raw term frequency?
=>TF–IDF reduces the importance of common words and highlights important, unique words in a document. Unlike raw TF, it gives more meaningful representation of text.

#2. Which terms receive higher importance?
=> Words that appear frequently in a document but rarely across all documents get higher importance (e.g., “chased”, “barked”).

#3. Effect of adding more documents
=> If more documents are added, common words get lower importance, while rare words become more important. TF–IDF values adjust based on the corpus.

A Comprehensive Guide on Continuous Bag of Words Model.

 Step 1: Corpus (after stop-word removal)

In [ ]:
docs = {
    "D1": ["cat", "chased", "mouse"],
    "D2": ["dog", "barked", "cat"],
    "D3": ["mouse", "ran", "cat"]
}

Step 2: Vocabulary + Indexing


In [ ]:
# FIXED vocabulary creation
vocab = sorted(set(
    word
    for doc in docs.values()
    for word in (doc if isinstance(doc, list) else doc.lower().replace('.', '').split())
))

print("Vocabulary:", vocab)

word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}

print("\nWord to Index:")
print(word_to_index)

Vocabulary: ['barked', 'cat', 'chased', 'dog', 'mouse', 'ran']

Word to Index:
{'barked': 0, 'cat': 1, 'chased': 2, 'dog': 3, 'mouse': 4, 'ran': 5}


 Step 3: One-Hot Encoding

In [ ]:
import numpy as np

vocab_size = len(vocab)

def one_hot(index):
    vec = np.zeros(vocab_size)
    vec[index] = 1
    return vec

# Example
print("\nOne-hot for 'cat':", one_hot(word_to_index["cat"]))


One-hot for 'cat': [0. 1. 0. 0. 0. 0.]


Step 4: Create Context–Target Pairs (CBOW)

In [ ]:
def generate_cbow_data(docs, word_to_index, window_size=1):
    data = []

    for doc_name, words in docs.items():
        for i, target_word in enumerate(words):

            context = []

            # Left context
            for j in range(i - window_size, i):
                if j >= 0:
                    context.append(word_to_index[words[j]])

            # Right context
            for j in range(i + 1, i + window_size + 1):
                if j < len(words):
                    context.append(word_to_index[words[j]])

            target = word_to_index[target_word]

            data.append((context, target))

    return data

cbow_data = generate_cbow_data(docs, word_to_index)

Step 5: Print Lookup Table (ALL DOCS)

In [ ]:
print("\nCBOW Context-Target Pairs:\n")

for context, target in cbow_data:
    context_words = [index_to_word[i] for i in context]
    target_word = index_to_word[target]

    print(f"Context: {context_words} -> Target: {target_word}")


CBOW Context-Target Pairs:

Context: ['chased'] -> Target: cat
Context: ['cat', 'mouse'] -> Target: chased
Context: ['chased'] -> Target: mouse
Context: ['barked'] -> Target: dog
Context: ['dog', 'cat'] -> Target: barked
Context: ['barked'] -> Target: cat
Context: ['ran'] -> Target: mouse
Context: ['mouse', 'cat'] -> Target: ran
Context: ['ran'] -> Target: cat


#1. How does CBOW handle this?
=>CBOW handles this by learning from multiple contexts of the same word.
Each time the word “cat” appears, it uses the surrounding context words to predict it. The model averages the context vectors and updates the word embedding based on all occurrences.

As a result, the final embedding of “cat” becomes a combination of all its different contexts, capturing its overall meaning.

#2. Connection to Distributional Hypothesis
=>The Distributional Hypothesis states that “a word is known by the company it keeps.”

CBOW follows this idea by learning word meaning from surrounding words. Since “cat” appears with different context words like “chased,” “barked,” “ran”, the model learns its meaning based on these relationships.

Thus, CBOW creates embeddings where words with similar contexts have similar meanings, directly supporting the Distributional Hypothesis.


"word2vec" - CBOW Architecture Design and Training.

Step 1: Setup Vocabulary & Index

In [ ]:
import numpy as np

# Vocabulary
vocab = ['barked', 'cat', 'chased', 'dog', 'mouse', 'ran']
word_to_index = {w: i for i, w in enumerate(vocab)}
index_to_word = {i: w for w, i in word_to_index.items()}

V = len(vocab)   # vocab size = 6
N = 4            # embedding size (given)

Step 2: One-hot Function

In [ ]:
def one_hot(word):
    vec = np.zeros(V)
    vec[word_to_index[word]] = 1
    return vec

 Step 3: Define Input Context

In [ ]:
context_words = ["cat", "mouse"]
target_word = "chased"

# One-hot vectors
x1 = one_hot("cat")
x2 = one_hot("mouse")

print("x1 (cat):", x1)
print("x2 (mouse):", x2)

x1 (cat): [0. 1. 0. 0. 0. 0.]
x2 (mouse): [0. 0. 0. 0. 1. 0.]


Step 4: Define Weight Matrices (Given)

In [ ]:
W_input = np.array([
    [0.1,  0.2, -0.1,  0.0],   # barked
    [0.4,  0.5, -0.1,  0.2],   # cat
    [-0.2, 0.3,  0.2, -0.1],   # chased
    [0.3, -0.1,  0.1,  0.0],   # dog
    [0.5, -0.4,  0.1,  0.1],   # mouse
    [-0.3, 0.0, -0.2,  0.3]    # ran
])

In [ ]:
W_hidden = np.array([
    [ 0.2, -0.1,  0.3,  0.1,  0.4,  0.0],
    [ 0.1,  0.3, -0.2, -0.1,  0.2,  0.5],
    [-0.4,  0.5,  0.1, -0.2, -0.1,  0.3],
    [ 0.1, -0.2, -0.4,  0.3, -0.3,  0.2]
])

Step 5: Hidden Layer Computation

In [ ]:
h1 = np.dot(x1, W_input)   # embedding for "cat"
h2 = np.dot(x2, W_input)   # embedding for "mouse"

print("Embedding (cat):", h1)
print("Embedding (mouse):", h2)

h = (h1 + h2) / 2
print("Hidden layer (average):", h)

Embedding (cat): [ 0.4  0.5 -0.1  0.2]
Embedding (mouse): [ 0.5 -0.4  0.1  0.1]
Hidden layer (average): [0.45 0.05 0.   0.15]


Step 6: Output Layer

In [ ]:
u = np.dot(h, W_hidden)
print("Raw output (u):", u)

Raw output (u): [ 0.11  -0.06   0.065  0.085  0.145  0.055]


Step 7: Softmax (Final Probabilities)

In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x))  # stability
    return exp_x / np.sum(exp_x)

y_pred = softmax(u)

print("\nPredicted probabilities:")
for i, prob in enumerate(y_pred):
    print(index_to_word[i], ":", round(prob, 4))


Predicted probabilities:
barked : 0.1737
cat : 0.1465
chased : 0.1661
dog : 0.1694
mouse : 0.1799
ran : 0.1644


 Step 8: Predicted Word

In [ ]:
predicted_index = np.argmax(y_pred)
print("\nPredicted word:", index_to_word[predicted_index])
print("Actual target:", target_word)


Predicted word: mouse
Actual target: chased


Step 9: Show Output Layer Size (VERY IMPORTANT FOR THEORY)

In [24]:
print("Vocabulary size (Output neurons):", V)

Vocabulary size (Output neurons): 6


Step 10: Show Target One-Hot Vector

In [25]:
y_true = one_hot(target_word)

print("\nTarget one-hot vector:", y_true)


Target one-hot vector: [0. 0. 1. 0. 0. 0.]


Step 11: Compare Prediction vs Actual

In [26]:
print("\nPredicted probabilities:", y_pred)
print("Actual one-hot:", y_true)


Predicted probabilities: [0.17369926 0.14654395 0.16605605 0.16941061 0.17988637 0.16440377]
Actual one-hot: [0. 0. 1. 0. 0. 0.]


Step 12: Compute Loss

In [27]:
loss = -np.sum(y_true * np.log(y_pred + 1e-9))
print("\nLoss:", loss)


Loss: 1.7954298862783649


Step 13: Show Why Output Layer Works

In [28]:
print("\nOutput Layer Explanation:")
print("Input to output layer shape:", h.shape)
print("Weight matrix shape:", W_hidden.shape)
print("Output logits shape:", u.shape)


Output Layer Explanation:
Input to output layer shape: (4,)
Weight matrix shape: (4, 6)
Output logits shape: (6,)


In [29]:
print("\n--- After Training Concept ---")
print("Before training → random prediction:", index_to_word[np.argmax(y_pred)])

print("After training → model learns correct mapping")
print("Expected prediction →", target_word)


--- After Training Concept ---
Before training → random prediction: mouse
After training → model learns correct mapping
Expected prediction → chased


 Step 15: Extract Word Embedding

In [30]:
print("\n--- Word Embeddings (W_input) ---")

for word, idx in word_to_index.items():
    print(word, ":", W_input[idx])


--- Word Embeddings (W_input) ---
barked : [ 0.1  0.2 -0.1  0. ]
cat : [ 0.4  0.5 -0.1  0.2]
chased : [-0.2  0.3  0.2 -0.1]
dog : [ 0.3 -0.1  0.1  0. ]
mouse : [ 0.5 -0.4  0.1  0.1]
ran : [-0.3  0.  -0.2  0.3]


Step 16: Similarity Check

In [31]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print("\nSimilarity between cat & dog:",
      cosine_similarity(W_input[word_to_index["cat"]],
                        W_input[word_to_index["dog"]]))


Similarity between cat & dog: 0.2667325346846321
